# 36B The interface, your ten questions, and what it costs
The app is thin: it calls the same graph. Your ten questions are the demonstration.

In [ ]:
import sys, json, pathlib, tempfile
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "askpanel").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from askpanel import audit, catalog, clock, db, econ, gate, llm, nodes, questions, standin, tools
P = yaml.safe_load(open(ROOT / "params.yaml"))
PANEL = db.load_panel(ROOT / P["data"]["panel"])
TMP = pathlib.Path(tempfile.mkdtemp())
BASE = db.build_db(PANEL, TMP / "base.sqlite")          # tables only: works before you write the view
print(f"{len(PANEL)} panel rows, {PANEL['adsh'].nunique()} filings; scratch database {BASE.name}")

In [ ]:
print("Coverage rules:", questions.MINIMUM)
for p in questions.validate(questions.load(ROOT / "questions/questions.example.yaml")):
    print("  ", p)

In [ ]:
led = audit.Ledger(clock=clock.clock)
T = tools.local_tools(BASE, led, tables=("filings", "holdings"))
N, a, b = nodes.make_nodes(standin.FixtureSQLModel(), T, clock.clock, allowed={"filings", "holdings"})
outs = []
for q in standin.FIXTURE:
    s = nodes.run_plain(nodes.new_state(q), N, a, b)
    outs.append({"sql": s["query"]["sql"], "usd": s["usd"], "trace": s["trace"]})
print(econ.summary(outs))
print(econ.hours_saved_table(P["economics"]["analyst_minutes_per_question"] / 60,
                             P["economics"]["analyst_rate_usd"], tasks=10))

## The app, in outline
```python
if q := st.chat_input("Ask about the BDC holdings"):
    cfg = {"configurable": {"thread_id": st.session_state.thread}}
    prev = app.get_state(cfg).values.get("history", [])
    s = app.invoke(nodes.new_state(q, prev), cfg)
    st.write(s["answer"]); st.code(s["query"]["sql"], language="sql")
```
Cache the graph and the MCP session with `st.cache_resource`: Streamlit reruns the script on every input.

**Your turn (Parts 4 to 6, 30 points).**
1. `app/streamlit_app.py`: question box, the answer, the SQL, a results table, the cost, and conversation history.
2. `questions/questions.yaml`: ten questions on **your real panel** (`data.use_real: true`), meeting the coverage rules. Run `python agent/run.py --mode record` once, then `--mode replay` to confirm. Commit `outputs/questions/` and `outputs/cache/`.
3. `outputs/economics.md` (five lines, with numbers) and `outputs/failures.md` (three failures, each with what fixed it).
4. `PYTHONPATH=src python -m askpanel.check`: 12/12. Then record the demo video (10 minutes or less).